# 04 Clinical RAG, grounded and guarded

Retrieval augmented generation (RAG) is how you make an assistant answer from *your* documents instead of from a model's memory. This notebook builds a tiny, fully offline RAG pipeline over a small set of clinical guideline snippets, so it runs in Colab with no API key.

You will see the three ideas that make RAG safe for clinical use:
1. **Retrieve** the most relevant source passages for a question.
2. **Ground** the answer in those passages, with a citation.
3. **Guard**: if nothing relevant is found, the assistant abstains instead of guessing.

This is a teaching pipeline. In production you would swap TF-IDF for real embeddings and the template for a language model, but the structure and the safety logic stay the same.

## 1. A small knowledge base

In a real system these would be your department's SOPs, formulary, and guidelines, split into chunks. Here we use a handful of short, illustrative snippets. None of this is medical advice; it exists to demonstrate the pipeline.

In [ ]:
knowledge_base = [
    {"id": "SEPSIS-01", "text": "Adult sepsis screening uses the qSOFA score: respiratory rate 22 or higher, altered mentation, and systolic blood pressure 100 mmHg or lower. Two or more suggests higher risk and prompts escalation."},
    {"id": "SEPSIS-02", "text": "In suspected sepsis, obtain blood cultures before antibiotics where it does not delay treatment, and start broad-spectrum antibiotics within one hour of recognition."},
    {"id": "DM-01", "text": "HbA1c of 6.5 percent or higher on two occasions is consistent with diabetes mellitus. Fasting plasma glucose of 126 mg/dL or higher also meets the threshold."},
    {"id": "DM-02", "text": "First-line pharmacotherapy for type 2 diabetes in most adults is metformin, alongside lifestyle change, unless contraindicated by renal impairment."},
    {"id": "HTN-01", "text": "Hypertension in adults is commonly defined as office blood pressure of 140/90 mmHg or higher confirmed on repeat measurement."},
    {"id": "VTE-01", "text": "Assess venous thromboembolism risk on admission. Pharmacological prophylaxis is considered for medical inpatients with raised risk and no contraindication to anticoagulation."}
]

documents = [d["text"] for d in knowledge_base]
print(f"Knowledge base: {len(documents)} passages")

## 2. Index the passages

We turn each passage into a vector with TF-IDF, so we can measure how close a question is to each passage. In a real system this is where sentence embeddings go.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

vectorizer = TfidfVectorizer(stop_words="english")
doc_vectors = vectorizer.fit_transform(documents)
print("Indexed. Vocabulary size:", len(vectorizer.vocabulary_))

## 3. Retrieve

For a question, find the closest passages by cosine similarity, and keep the best one with its score.

In [ ]:
def retrieve(question, k=2):
    q_vec = vectorizer.transform([question])
    sims = cosine_similarity(q_vec, doc_vectors)[0]
    top = np.argsort(sims)[::-1][:k]
    return [(knowledge_base[i]["id"], documents[i], float(sims[i])) for i in top]

for src_id, text, score in retrieve("What HbA1c level means diabetes?"):
    print(f"{src_id}  (score {score:.2f})  {text[:70]}...")

## 4. Ground the answer, and guard it

This is the safety logic. If the best match is too weak, the assistant refuses to answer rather than inventing one. When it does answer, it returns the source passage and its id so a clinician can check it. A real system would pass the retrieved passage to a language model with the instruction *answer only from this source*; the guardrail is identical.

In [ ]:
RELEVANCE_THRESHOLD = 0.15  # below this, we do not trust the match

def answer(question):
    hits = retrieve(question, k=1)
    src_id, text, score = hits[0]
    if score < RELEVANCE_THRESHOLD:
        return {
            "answer": "I do not have a source that covers this. Please consult the full guideline or a clinician.",
            "source": None,
            "confidence": round(score, 2),
        }
    return {
        "answer": text,
        "source": src_id,
        "confidence": round(score, 2),
    }

import json
for q in [
    "What is first line treatment for type 2 diabetes?",
    "How do I screen for sepsis in adults?",
    "What is the best treatment for a broken arm?",  # not in the KB, should abstain
]:
    print("Q:", q)
    print(json.dumps(answer(q), indent=2, ensure_ascii=False))
    print("-" * 60)

## 5. What you just built, and what changes in production

You built the full shape of a safe clinical assistant: retrieve, ground, cite, and abstain when unsure. The third example shows the guardrail working, the assistant declines a question its sources do not cover.

To take this to production:
- Replace TF-IDF with a sentence-embedding model and a vector database.
- Replace the passage-return with a language model instructed to answer only from the retrieved passages.
- Keep the threshold-based abstention and the citation. These are what make it safe.
- Add a human-in-the-loop checkpoint before anything with clinical consequence, and log every interaction for audit.

## Exercise

1. Add three passages from a guideline you know, and ask questions that should and should not be answerable.
2. Tune `RELEVANCE_THRESHOLD`. What happens to false answers as you raise it? What do you lose?
3. Sketch where a clinician would sign off if this assistant drafted a reply to a patient on Line.